# Assignment 1: Market Clearing (System Perspective)

### Import external libraries

In [40]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import network
import networkx as nx
from network import initialize_network, system_demand, LINES


## Step 0: Build a Relevant Case Study

Please select an electric power network from the following options:
1. IEEE 24-bus reliability test system: link.
2. IEEE reliability test system (2019 update): link.
3. IEEE power systems test cases (various cases with 14, 30, 57, 118, and 300 buses): link.
4. Case studies available in the open-source Julia platform PowerModels.jl: link.

You are also free to choose another case study. If some data is missing, please select reason-
able arbitrary values. For technical details on conventional generators and transmission lines, this link may be helpful (it corresponds to the IEEE 24-bus case study, but similar data can be used for other cases).

### IEEE 24-bus reliability test system

In [41]:
consumers, generators = initialize_network()
print(generators)
print(consumers)
print("Network initialized successfully.")

[Generator(unit=1, node=1), Generator(unit=2, node=2), Generator(unit=3, node=7), Generator(unit=4, node=13), Generator(unit=5, node=15), Generator(unit=6, node=15), Generator(unit=7, node=16), Generator(unit=8, node=18), Generator(unit=9, node=21), Generator(unit=10, node=22), Generator(unit=11, node=23), Generator(unit=12, node=23), Generator(unit=13, node=3), Generator(unit=14, node=5), Generator(unit=15, node=7), Generator(unit=16, node=16), Generator(unit=17, node=21), Generator(unit=18, node=23)]
[Consumer(load=1, node=1, share=0.038), Consumer(load=2, node=2, share=0.034), Consumer(load=3, node=3, share=0.063), Consumer(load=4, node=4, share=0.026), Consumer(load=5, node=5, share=0.025), Consumer(load=6, node=6, share=0.048), Consumer(load=7, node=7, share=0.044), Consumer(load=8, node=8, share=0.06), Consumer(load=9, node=9, share=0.061), Consumer(load=10, node=10, share=0.068), Consumer(load=11, node=13, share=0.093), Consumer(load=12, node=14, share=0.068), Consumer(load=13, 

### Additional Assumptions


*   Assume that the price bids of all producers are non-negative and equal to their marginal production cost. In particular, the production cost of renewable units is assumed to be zero. Additionally, these units offer their forecasted capacity, meaning their offer quantities vary over time.

*   For the bid price of price-elastic demands, use comparatively high values (relative to the generation cost of conventional units) to ensure that most demands are supplied. For inspiration, check the real bid price data in Nord Pool [link].
*   A potential source for wind power forecast data is available at this link (you may nor- malize the data to fit your case study). Another potential source for the renewable power generation data is renewables.ninja.
*   For transmission lines, you may assume a uniform reactance for all lines (e.g., 0.002 p.u., leading to a susceptance of 500 p.u.).

## Step 2 – Copper-Plate Market Clearing with Multiple Hours

In this step, the single-hour market-clearing model is extended to a **multi-period setting** covering **24 hours**. A new time index is introduced:

$$
t = 1,2,\dots,24
$$

Some input parameters vary across hours, such as:

- wind power generation
- demand bid prices
- demand bid quantities

Demand bids should be generated such that they are **higher during peak hours**.

In addition, a **large-scale energy storage unit** (for example pumped hydro storage) is introduced in order to model **intertemporal constraints**. The objective of the market-clearing problem is now to **maximize total social welfare over the full 24-hour horizon**. :contentReference[oaicite:0]{index=0}

### Storage model

We use lowercase symbols for **decision variables** and uppercase or Greek symbols for **parameters**.

#### Storage variables

- $p^{ch}_t$: charging power in hour $t$ (MW)
- $p^{dis}_t$: discharging power in hour $t$ (MW)
- $e_t$: stored energy in hour $t$ (MWh)

#### Storage constraints

##### Charging power limit

$$
0 \leq p^{ch}_t \leq P^{ch}, \quad \forall t
$$

##### Discharging power limit

$$
0 \leq p^{dis}_t \leq P^{dis}, \quad \forall t
$$

##### Energy capacity limit

$$
0 \leq e_t \leq E, \quad \forall t
$$

##### Intertemporal energy balance

$$
e_t = e_{t-1} + p^{ch}_t \eta^{ch} - \frac{p^{dis}_t}{\eta^{dis}}, \quad \forall t
$$

where:

- $\eta^{ch}$ is the charging efficiency
- $\eta^{dis}$ is the discharging efficiency

#### Storage parameters

- $P^{ch}$: maximum charging power
- $P^{dis}$: maximum discharging power
- $E$: energy storage capacity
- $\eta^{ch}$: charging efficiency
- $\eta^{dis}$: discharging efficiency

To avoid simultaneous charging and discharging without introducing binary variables, it is assumed that:

$$
0 < \eta^{ch} < \eta^{dis} < 1
$$

### Additional assumptions

- The storage size should be of the same order of magnitude as the capacities of conventional generators.
- The storage unit submits **zero bid and zero offer prices** to the market.
- Therefore, the presence of storage does **not change the objective function directly**.
- In reality, storage bidding would involve opportunity costs, but this is neglected here for simplicity.

### Required market-clearing outcomes

The following outputs should be computed over the 24-hour horizon:

1. **Hourly market-clearing prices** under a uniform pricing scheme
2. **Total operating cost** over 24 hours
3. **Total social welfare** over 24 hours
4. **Total profit of each producer** over 24 hours:
   - conventional generators
   - wind farms
5. **Total profit of the storage unit** over 24 hours

### Further analysis

#### 1. Comparison of prices with and without storage

Compare the hourly market-clearing prices obtained:

- without storage
- with storage

Discuss whether storage smooths prices across hours or creates any visible arbitrage pattern.

#### 2. Sensitivity analysis

Perform a sensitivity analysis by changing:

- the storage energy capacity $E$
- the charging power limit $P^{ch}$
- the discharging power limit $P^{dis}$

Then explain how and why these changes affect:

- market-clearing prices
- storage operation
- total social welfare

#### 3. Price formation in the presence of storage

Investigate whether the market-clearing price in each hour is still equal to the offer price of the marginal producer when storage is present.

Discuss your numerical findings and explain how the intertemporal coupling introduced by storage may affect hourly price formation.

In [42]:
# -*- coding: utf-8 -*-
"""
Copper-plate market clearing (24h) with conventional generators, wind, demand bids,
and battery storage. Objective: maximize social welfare.

Translated from Julia/JuMP + HiGHS to Python + Gurobi (gurobipy).
"""

import numpy as np
from gurobipy import Model, GRB, quicksum

# ============================================================
# SECTION 1 – INPUT DATA
# ============================================================

T  = 24   # hours
nG = 12   # conventional generators
nW = 6    # wind farms
nD = 17   # demand loads

# -------------------------------
# 1a. CONVENTIONAL GENERATORS
# -------------------------------
Pmax = [152.0, 152.0, 350.0, 591.0,  60.0, 155.0,
        155.0, 400.0, 400.0, 300.0, 310.0, 350.0]

RU = [120.0, 120.0, 350.0, 240.0,  60.0, 155.0,
      155.0, 280.0, 280.0, 300.0, 180.0, 240.0]

RD = [120.0, 120.0, 350.0, 240.0,  60.0, 155.0,
      155.0, 280.0, 280.0, 300.0, 180.0, 240.0]

Cg = [13.32, 13.32, 20.70, 20.93, 26.11, 10.52,
      10.52,  6.02,  5.47,  0.00, 10.52, 10.89]

Pini = [ 76.0,  76.0,   0.0,   0.0,   0.0,   0.0,
        124.0, 240.0, 240.0, 240.0, 248.0, 280.0]

# -------------------------------
# 1b. DEMAND / LOADS
# -------------------------------
system_demand = [1775.835, 1669.815, 1590.300, 1563.795, 1563.795, 1590.300,
                 1961.370, 2279.430, 2517.975, 2544.480, 2544.480, 2517.975,
                 2517.975, 2517.975, 2464.965, 2464.965, 2623.995, 2650.500,
                 2650.500, 2544.480, 2411.955, 2199.915, 1934.865, 1669.815]

load_fraction = [3.8, 3.4, 6.3, 2.6, 2.5, 4.8, 4.4,
                 6.0, 6.1, 6.8, 9.3, 6.8, 11.1, 3.5,
                 11.7, 6.4, 4.5]

# PD[d,t] = fraction * system_demand[t]
PD = np.zeros((nD, T), dtype=float)
for d in range(nD):
    for t in range(T):
        PD[d, t] = (load_fraction[d] / 100.0) * system_demand[t]

Ud = [165.0, 155.0, 175.0, 150.0, 145.0, 170.0, 160.0, 180.0, 185.0,
      190.0, 210.0, 195.0, 220.0, 158.0, 215.0, 178.0, 162.0]

# -------------------------------
# 1c. WIND DATA (24x6)
# -------------------------------
wind_data = np.array([
    [384.46, 563.37, 362.61, 202.07, 558.26, 468.19],
    [334.14, 556.43, 527.93, 314.27, 548.84, 553.79],
    [392.11, 620.06, 533.58, 432.35, 607.21, 574.22],
    [320.72, 565.90, 619.15, 452.17, 609.68, 607.52],
    [511.10, 662.96, 686.13, 584.76, 663.70, 689.71],
    [670.20, 672.05, 709.74, 545.10, 641.83, 701.07],
    [732.58, 683.56, 720.57, 627.75, 670.87, 719.54],
    [715.88, 690.68, 710.55, 608.72, 658.43, 732.09],
    [816.48, 706.20, 714.52, 697.07, 703.08, 717.60],
    [863.17, 655.94, 681.27, 730.43, 615.06, 640.84],
    [834.68, 684.01, 707.80, 738.26, 631.36, 658.99],
    [809.60, 701.91, 671.20, 678.32, 589.42, 683.79],
    [779.70, 707.42, 633.47, 588.80, 583.27, 535.43],
    [737.25, 705.13, 685.57, 618.84, 605.68, 533.90],
    [720.23, 744.80, 691.89, 620.54, 661.68, 625.76],
    [745.21, 763.06, 743.45, 646.46, 724.46, 662.57],
    [682.32, 727.55, 695.04, 658.23, 678.63, 572.37],
    [656.48, 695.86, 657.82, 591.17, 657.69, 520.87],
    [734.26, 724.75, 772.78, 684.40, 753.31, 725.85],
    [724.07, 771.75, 802.11, 707.73, 731.72, 718.83],
    [736.49, 824.36, 835.11, 722.33, 723.93, 661.45],
    [631.56, 782.89, 825.19, 729.68, 717.15, 632.22],
    [624.39, 815.08, 821.35, 736.51, 669.55, 628.15],
    [689.31, 751.98, 772.77, 709.17, 567.82, 552.70],
], dtype=float)

# -------------------------------
# 1d. BATTERY
# -------------------------------
Cap_BS   = 700.0
Ch_rate  = 140.0
Dis_rate = 140.0
eta_c    = 0.885
eta_d    = 0.885
alpha    = 0.0

# ============================================================
# SECTION 2 – BUILD + SOLVE BASE MODEL
# ============================================================

def build_and_solve(Cap_BS_case=Cap_BS, Ch_rate_case=Ch_rate, Dis_rate_case=Dis_rate, silent=True):
    m = Model("copper_plate_market")
    if silent:
        m.setParam("OutputFlag", 0)
    m.ModelSense = GRB.MAXIMIZE

    # Decision variables
    pG   = m.addVars(nG, T, lb=0.0, name="pG")
    pW   = m.addVars(nW, T, lb=0.0, name="pW")
    pDsv = m.addVars(nD, T, lb=0.0, name="pD")   # served demand
    pCh  = m.addVars(T, lb=0.0, name="pCh")
    pDis = m.addVars(T, lb=0.0, name="pDis")
    SOC  = m.addVars(T, lb=0.0, name="SOC")

    # Objective: max social welfare
    m.setObjective(
        quicksum(Ud[d] * pDsv[d, t] for d in range(nD) for t in range(T))
        - quicksum(Cg[g] * pG[g, t] for g in range(nG) for t in range(T))
    )

    # Generator caps
    for g in range(nG):
        for t in range(T):
            m.addConstr(pG[g, t] <= Pmax[g], name=f"gen_cap[{g},{t}]")

    # Demand bid caps
    for d in range(nD):
        for t in range(T):
            m.addConstr(pDsv[d, t] <= PD[d, t], name=f"dem_cap[{d},{t}]")

    # Wind availability (with the same cap min(200, wind_data))
    for w in range(nW):
        for t in range(T):
            m.addConstr(pW[w, t] <= min(200.0, float(wind_data[t, w])), name=f"wind_cap[{w},{t}]")

    # Power balance (store constraints to read MCP duals)
    balance = []
    for t in range(T):
        c = m.addConstr(
            quicksum(pG[g, t] for g in range(nG))
            + quicksum(pW[w, t] for w in range(nW))
            + pDis[t]
            ==
            quicksum(pDsv[d, t] for d in range(nD))
            + pCh[t],
            name=f"balance[{t}]"
        )
        balance.append(c)

    # Ramp constraints t>=2
    for g in range(nG):
        for t in range(1, T):
            m.addConstr(pG[g, t] - pG[g, t-1] <=  RU[g], name=f"ramp_up[{g},{t}]")
            m.addConstr(pG[g, t] - pG[g, t-1] >= -RD[g], name=f"ramp_dn[{g},{t}]")

    # Ramp constraints at t=1 using Pini
    for g in range(nG):
        m.addConstr(pG[g, 0] - Pini[g] <=  RU[g], name=f"ramp_up_ini[{g}]")
        m.addConstr(pG[g, 0] - Pini[g] >= -RD[g], name=f"ramp_dn_ini[{g}]")

    # SOC cap (store to read duals in sensitivity)
    soc_cap = []
    for t in range(T):
        soc_cap.append(m.addConstr(SOC[t] <= Cap_BS_case, name=f"soc_cap[{t}]"))

    # SOC dynamics
    for t in range(1, T):
        m.addConstr(
            SOC[t] == SOC[t-1] + eta_c * pCh[t] - (1.0 / eta_d) * pDis[t],
            name=f"soc_dyn[{t}]"
        )
    m.addConstr(
        SOC[0] == alpha + eta_c * pCh[0] - (1.0 / eta_d) * pDis[0],
        name="soc_dyn[0]"
    )

    # Charge/discharge power limits (store to read duals)
    ch_cap = []
    dis_cap = []
    for t in range(T):
        ch_cap.append(m.addConstr(pCh[t] <= Ch_rate_case, name=f"ch_cap[{t}]"))
        dis_cap.append(m.addConstr(pDis[t] <= Dis_rate_case, name=f"dis_cap[{t}]"))

    # Solve
    m.optimize()

    return m, (pG, pW, pDsv, pCh, pDis, SOC), (balance, soc_cap, ch_cap, dis_cap)

# ---------- Solve base case ----------
model, vars_, constrs_ = build_and_solve(Cap_BS, Ch_rate, Dis_rate, silent=True)
pG, pW, pDsv, pCh, pDis, SOC = vars_
balance, soc_cap, ch_cap, dis_cap = constrs_

print("\n=== SOLVER STATUS ===")
status_map = {
    GRB.OPTIMAL: "OPTIMAL",
    GRB.INFEASIBLE: "INFEASIBLE",
    GRB.UNBOUNDED: "UNBOUNDED",
    GRB.INF_OR_UNBD: "INF_OR_UNBD",
}
print("Status:", status_map.get(model.Status, str(model.Status)))

if model.Status != GRB.OPTIMAL:
    raise RuntimeError("Model did not solve to optimality.")

# ============================================================
# SECTION 4 – RESULTS
# ============================================================

# MCP = dual of balance constraints (Pi in Gurobi)
mcp = np.array([balance[t].Pi for t in range(T)], dtype=float)

SW = float(model.ObjVal)
total_gen_cost = sum(Cg[g] * pG[g, t].X for g in range(nG) for t in range(T))

gen_profit = []
for g in range(nG):
    prof = sum((mcp[t] - Cg[g]) * pG[g, t].X for t in range(T))
    gen_profit.append(prof)

wind_profit = []
for w in range(nW):
    prof = sum(mcp[t] * pW[w, t].X for t in range(T))
    wind_profit.append(prof)

bat_profit = sum(mcp[t] * (pDis[t].X - pCh[t].X) for t in range(T))

demand_utility = []
for d in range(nD):
    util = sum(pDsv[d, t].X * (Ud[d] - mcp[t]) for t in range(T))
    demand_utility.append(util)

# ============================================================
# SECTION 5 – PRINT RESULTS (same style)
# ============================================================

print("\n====================================================")
print("  COPPER-PLATE MARKET CLEARING RESULTS (24 h)")
print("====================================================")
print(f"\nTotal Social Welfare   : ${SW:,.2f}")
print(f"Total Generation Cost  : ${total_gen_cost:,.2f}")

print("\n--- Hourly Market Clearing Prices ---")
print(" Hour | MCP [$/MWh] | SysDemand [MW] | TotalGen [MW] | SOC [MWh]")
print("------+------------+----------------+---------------+----------")
for t in range(T):
    gen_t = sum(pG[g, t].X for g in range(nG)) + sum(pW[w, t].X for w in range(nW))
    print(f" {t+1:3d}  |  {mcp[t]:9.4f} |   {system_demand[t]:10.3f}   |  {gen_t:10.3f}   |  {SOC[t].X:6.2f}")

print("\n--- Profit of each producer [$] ---")
print("(Generators 1-12 conventional | Generators 13-18 wind farms)\n")

all_profit = list(gen_profit) + list(wind_profit)  # length 18

# Row 1: 1-9
print(f"  {'Generator':<13}", end="")
for i in range(1, 10):
    print(f" {i:8d} |", end="")
print()
print(f"  {'Profit [$]':<13}", end="")
for i in range(1, 10):
    print(f" {int(round(all_profit[i-1])):8d} |", end="")
print("\n")

# Row 2: 10-18
print(f"  {'Generator':<13}", end="")
for i in range(10, 19):
    print(f" {i:8d} |", end="")
print()
print(f"  {'Profit [$]':<13}", end="")
for i in range(10, 19):
    print(f" {int(round(all_profit[i-1])):8d} |", end="")
print()

print(f"\n\n--- Battery Profit : ${int(round(bat_profit))} ---")

print("\n--- Demand Utility [$] ---")
for d in range(nD):
    print(f"  Load {d+1:<2d}: ${int(round(demand_utility[d]))}")

print("\n====================================================")
print("Cross-check: SW = producer profits + consumer utility")
print(f"  Sum gen profits  : ${int(round(sum(gen_profit)))}")
print(f"  Sum wind profits : ${int(round(sum(wind_profit)))}")
print(f"  Battery profit   : ${int(round(bat_profit))}")
print(f"  Sum utilities    : ${int(round(sum(demand_utility)))}")
print(f"  Total            : ${int(round(sum(gen_profit) + sum(wind_profit) + bat_profit + sum(demand_utility)))}"
      f"  (objective = ${int(round(SW))})")
print("====================================================")

# ============================================================
# SECTION SA – SENSITIVITY ANALYSIS (OAT)
# ============================================================

def solve_case(Cap_BS_case: float, Ch_rate_case: float, Dis_rate_case: float):
    m, (pG_, pW_, pD_, pCh_, pDis_, SOC_), (balance_, soc_cap_, ch_cap_, dis_cap_) = build_and_solve(
        Cap_BS_case, Ch_rate_case, Dis_rate_case, silent=True
    )

    if m.Status != GRB.OPTIMAL:
        return dict(ok=False, term=m.Status)

    mcp_ = np.array([balance_[t].Pi for t in range(T)], dtype=float)
    SW_ = float(m.ObjVal)
    total_gen_cost_ = sum(Cg[g] * pG_[g, t].X for g in range(nG) for t in range(T))
    bat_profit_ = sum(mcp_[t] * (pDis_[t].X - pCh_[t].X) for t in range(T))

    soc_vals = np.array([SOC_[t].X for t in range(T)], dtype=float)
    soc_max_ = float(np.max(soc_vals))
    soc_end_ = float(soc_vals[-1])
    total_charge_ = float(sum(pCh_[t].X for t in range(T)))
    total_dis_ = float(sum(pDis_[t].X for t in range(T)))

    # Dual diagnostics (sum over t)
    sum_soc_duals_ = float(sum(soc_cap_[t].Pi for t in range(T)))
    sum_ch_duals_  = float(sum(ch_cap_[t].Pi for t in range(T)))
    sum_dis_duals_ = float(sum(dis_cap_[t].Pi for t in range(T)))

    return dict(
        ok=True,
        term=m.Status,
        Cap_BS=Cap_BS_case,
        Ch_rate=Ch_rate_case,
        Dis_rate=Dis_rate_case,
        SW=SW_,
        total_gen_cost=total_gen_cost_,
        bat_profit=bat_profit_,
        mcp_avg=float(np.mean(mcp_)),
        mcp_max=float(np.max(mcp_)),
        soc_max=soc_max_,
        soc_end=soc_end_,
        total_charge=total_charge_,
        total_dis=total_dis_,
        sum_soc_duals=sum_soc_duals_,
        sum_ch_duals=sum_ch_duals_,
        sum_dis_duals=sum_dis_duals_,
    )

factors = [0.5, 0.75, 1.0, 1.25, 1.5]
Cap0, Ch0, Dis0 = float(Cap_BS), float(Ch_rate), float(Dis_rate)

results = []

# Vary Cap only
for f in factors:
    results.append(solve_case(Cap0*f, Ch0, Dis0))

# Vary Ch only
for f in factors:
    results.append(solve_case(Cap0, Ch0*f, Dis0))

# Vary Dis only
for f in factors:
    results.append(solve_case(Cap0, Ch0, Dis0*f))

print("\n====================================================")
print("  SENSITIVITY RESULTS (OAT): Cap_BS, Ch_rate, Dis_rate")
print("====================================================")
print(" Case | Cap_BS | Ch_rate | Dis_rate |    SW     | GenCost  | BatProf | MCPavg | MCPmax | SOCmax")
print("------+--------+---------+----------+----------+----------+---------+--------+--------+-------")

for idx, r in enumerate(results, start=1):
    if not r.get("ok", False):
        print(f" {idx:4d} | (infeasible/failed) status={r.get('term')}")
        continue
    print(f" {idx:4d} | {r['Cap_BS']:6.1f} | {r['Ch_rate']:7.1f} | {r['Dis_rate']:8.1f} |"
          f" {r['SW']:8.2f} | {r['total_gen_cost']:8.2f} | {r['bat_profit']:7.2f} |"
          f" {r['mcp_avg']:6.2f} | {r['mcp_max']:6.2f} | {r['soc_max']:6.1f}")

print("\n--- Dual (shadow price) diagnostics (sum over t) ---")
print("Larger magnitude typically => constraint binding more often => parameter more valuable.")
print("Case | sum_dual(SOC<=Cap) | sum_dual(pCh<=Ch) | sum_dual(pDis<=Dis)")
print("-----+---------------------+------------------+-------------------")

for idx, r in enumerate(results, start=1):
    if not r.get("ok", False):
        continue
    print(f"{idx:4d} | {r['sum_soc_duals']:19.4f} | {r['sum_ch_duals']:16.4f} | {r['sum_dis_duals']:17.4f}")


=== SOLVER STATUS ===
Status: OPTIMAL

  COPPER-PLATE MARKET CLEARING RESULTS (24 h)

Total Social Welfare   : $9,752,575.65
Total Generation Cost  : $108,931.69

--- Hourly Market Clearing Prices ---
 Hour | MCP [$/MWh] | SysDemand [MW] | TotalGen [MW] | SOC [MWh]
------+------------+----------------+---------------+----------
   1  |    -5.4700 |     1775.835   |    1915.835   |  123.90
   2  |    -5.4700 |     1669.815   |    1809.815   |  247.80
   3  |    -5.4700 |     1590.300   |    1730.300   |  371.70
   4  |    -5.4700 |     1563.795   |    1616.695   |  418.52
   5  |    -5.4700 |     1563.795   |    1703.795   |  542.42
   6  |    -5.4700 |     1590.300   |    1730.300   |  666.32
   7  |    -5.4700 |     1961.370   |    1999.430   |  700.00
   8  |    -6.5700 |     2279.430   |    2279.430   |  700.00
   9  |   -10.5200 |     2517.975   |    2517.975   |  700.00
  10  |   -10.5200 |     2544.480   |    2404.480   |  541.81
  11  |   -10.5200 |     2544.480   |    2544.480